# Notebook 52: Tight Trail Stop Optimization for Income

**Goal:** Find optimal trail stop for maximum trade frequency while maintaining profitability.

**Based on Notebook 51 findings:**
- 15% trail on 4H: 8 trades/yr, +2,368%
- Need to test tighter: 5%, 8%, 10%, 12%

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import vectorbt as vbt
from pathlib import Path

DATA_DIR = Path("../data")
HOURLY_DIR = DATA_DIR / "hourly"

## 1. Load and Prepare Data

In [ ]:
# Load hourly data
def load_hourly():
    price = pd.read_parquet(HOURLY_DIR / "price.parquet")
    sopr = pd.read_parquet(HOURLY_DIR / "sopr.parquet")
    sopr_sth = pd.read_parquet(HOURLY_DIR / "sopr_sth.parquet")
    sopr_lth = pd.read_parquet(HOURLY_DIR / "sopr_lth.parquet")
    realized_loss = pd.read_parquet(HOURLY_DIR / "realized_loss.parquet")
    
    df = price.rename(columns={"value": "price"}).set_index("time")
    df["sopr"] = sopr.set_index("time")["value"]
    df["sopr_sth"] = sopr_sth.set_index("time")["value"]
    df["sopr_lth"] = sopr_lth.set_index("time")["value"]
    df["realized_loss"] = realized_loss.set_index("time")["value"]
    
    return df

df_h1 = load_hourly()
print(f"Loaded {len(df_h1):,} hourly rows")

In [ ]:
# Resample to multiple timeframes
def resample_tf(df_h1, tf):
    df = pd.DataFrame()
    df["price"] = df_h1["price"].resample(tf).last()
    df["sopr"] = df_h1["sopr"].resample(tf).mean()
    df["sopr_sth"] = df_h1["sopr_sth"].resample(tf).mean()
    df["sopr_lth"] = df_h1["sopr_lth"].resample(tf).mean()
    df["realized_loss"] = df_h1["realized_loss"].resample(tf).sum()
    return df.dropna()

df_4h = resample_tf(df_h1, "4h")
df_8h = resample_tf(df_h1, "8h")
df_12h = resample_tf(df_h1, "12h")

print(f"4H: {len(df_4h):,} bars | 8H: {len(df_8h):,} bars | 12H: {len(df_12h):,} bars")

In [ ]:
# Add z-scores
def add_zscore(df, window):
    df = df.copy()
    df["rl_mean"] = df["realized_loss"].rolling(window=window, min_periods=window//2).mean()
    df["rl_std"] = df["realized_loss"].rolling(window=window, min_periods=window//2).std()
    df["rl_zscore"] = (df["realized_loss"] - df["rl_mean"]) / df["rl_std"]
    return df

df_4h = add_zscore(df_4h, 365 * 6)   # 2190 bars
df_8h = add_zscore(df_8h, 365 * 3)   # 1095 bars
df_12h = add_zscore(df_12h, 365 * 2) # 730 bars

# Filter to backtest period
START = "2019-01-01"
df_4h = df_4h[df_4h.index >= START].dropna()
df_8h = df_8h[df_8h.index >= START].dropna()
df_12h = df_12h[df_12h.index >= START].dropna()

years = (df_4h.index.max() - df_4h.index.min()).days / 365.25
bh_return = (df_4h["price"].iloc[-1] / df_4h["price"].iloc[0] - 1) * 100
print(f"\nBacktest: {years:.2f} years | B&H: {bh_return:+,.0f}%")

## 2. Helper Functions

In [ ]:
def get_metrics(pf, years):
    trades = pf.trades.records_readable
    if len(trades) == 0:
        return None
    
    total_return = pf.total_return() * 100
    trades_per_year = len(trades) / years
    
    durations = (trades["Exit Timestamp"] - trades["Entry Timestamp"]).dropna()
    avg_days = durations.mean().total_seconds() / 86400 if len(durations) > 0 else 0
    
    winning = trades[trades["PnL"] > 0]
    losing = trades[trades["PnL"] < 0]
    
    return {
        "return": total_return,
        "cagr": ((1 + total_return/100) ** (1/years) - 1) * 100,
        "sharpe": pf.sharpe_ratio(),
        "max_dd": pf.max_drawdown() * 100,
        "trades": len(trades),
        "trades_yr": trades_per_year,
        "avg_days": avg_days,
        "win_rate": (trades["PnL"] > 0).mean() * 100,
        "avg_win": winning["Return"].mean() * 100 if len(winning) > 0 else 0,
        "avg_loss": losing["Return"].mean() * 100 if len(losing) > 0 else 0,
        "profit_factor": abs(winning["PnL"].sum() / losing["PnL"].sum()) if len(losing) > 0 and losing["PnL"].sum() != 0 else np.inf,
        "expectancy": trades["Return"].mean() * 100,  # Average return per trade
    }

def run_strat002(df, freq, trail):
    cond = (df["sopr"] < 1) & (df["sopr_sth"] < 1) & (df["rl_zscore"] > 0.5)
    entry = cond & ~cond.shift(1).fillna(False)
    
    if entry.sum() == 0:
        return None
    
    return vbt.Portfolio.from_signals(
        close=df["price"],
        entries=entry,
        exits=None,
        sl_stop=trail,
        sl_trail=True,
        freq=freq,
        init_cash=10000,
        fees=0.001
    )

## 3. Fine-Grained Trail Stop Test (4H)

In [ ]:
# Test trail stops from 5% to 35% in 1% increments
trail_stops = [i/100 for i in range(5, 36, 1)]  # 5% to 35%

print("="*140)
print("4H TIMEFRAME: FINE-GRAINED TRAIL STOP ANALYSIS")
print("="*140)
print(f"\n{'Trail':>6} {'Return':>10} {'CAGR':>8} {'Sharpe':>7} {'MaxDD':>7} {'Trades':>7} {'Tr/Yr':>7} {'Days':>6} {'Win%':>6} {'AvgWin':>8} {'AvgLoss':>8} {'Expect':>8}")
print("-"*140)

results_4h = {}

for trail in trail_stops:
    pf = run_strat002(df_4h, "4h", trail)
    if pf:
        m = get_metrics(pf, years)
        results_4h[trail] = {"m": m, "pf": pf}
        print(f"{trail*100:>5.0f}% {m['return']:>+9,.0f}% {m['cagr']:>+7.1f}% {m['sharpe']:>7.2f} {m['max_dd']:>6.1f}% {m['trades']:>7} {m['trades_yr']:>7.1f} {m['avg_days']:>6.0f} {m['win_rate']:>5.0f}% {m['avg_win']:>+7.1f}% {m['avg_loss']:>+7.1f}% {m['expectancy']:>+7.1f}%")

In [ ]:
# Visualize trade-offs
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

trails = list(results_4h.keys())
returns = [results_4h[t]["m"]["return"] for t in trails]
trades = [results_4h[t]["m"]["trades"] for t in trails]
trades_yr = [results_4h[t]["m"]["trades_yr"] for t in trails]
sharpes = [results_4h[t]["m"]["sharpe"] for t in trails]
win_rates = [results_4h[t]["m"]["win_rate"] for t in trails]
avg_days = [results_4h[t]["m"]["avg_days"] for t in trails]

# Return vs Trail
axes[0,0].plot([t*100 for t in trails], returns, 'b-o', markersize=4)
axes[0,0].set_xlabel("Trail Stop (%)")
axes[0,0].set_ylabel("Total Return (%)")
axes[0,0].set_title("Return vs Trail Stop")
axes[0,0].grid(True, alpha=0.3)

# Trades/Year vs Trail
axes[0,1].plot([t*100 for t in trails], trades_yr, 'g-o', markersize=4)
axes[0,1].set_xlabel("Trail Stop (%)")
axes[0,1].set_ylabel("Trades per Year")
axes[0,1].set_title("Trade Frequency vs Trail Stop")
axes[0,1].grid(True, alpha=0.3)

# Win Rate vs Trail
axes[1,0].plot([t*100 for t in trails], win_rates, 'r-o', markersize=4)
axes[1,0].set_xlabel("Trail Stop (%)")
axes[1,0].set_ylabel("Win Rate (%)")
axes[1,0].set_title("Win Rate vs Trail Stop")
axes[1,0].axhline(y=50, color='black', linestyle='--', alpha=0.5)
axes[1,0].grid(True, alpha=0.3)

# Avg Hold Days vs Trail
axes[1,1].plot([t*100 for t in trails], avg_days, 'm-o', markersize=4)
axes[1,1].set_xlabel("Trail Stop (%)")
axes[1,1].set_ylabel("Avg Hold (Days)")
axes[1,1].set_title("Hold Duration vs Trail Stop")
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Fine-Grained Trail Stop Test (8H)

In [ ]:
print("="*140)
print("8H TIMEFRAME: FINE-GRAINED TRAIL STOP ANALYSIS")
print("="*140)
print(f"\n{'Trail':>6} {'Return':>10} {'CAGR':>8} {'Sharpe':>7} {'MaxDD':>7} {'Trades':>7} {'Tr/Yr':>7} {'Days':>6} {'Win%':>6} {'AvgWin':>8} {'AvgLoss':>8} {'Expect':>8}")
print("-"*140)

results_8h = {}

for trail in trail_stops:
    pf = run_strat002(df_8h, "8h", trail)
    if pf:
        m = get_metrics(pf, years)
        results_8h[trail] = {"m": m, "pf": pf}
        print(f"{trail*100:>5.0f}% {m['return']:>+9,.0f}% {m['cagr']:>+7.1f}% {m['sharpe']:>7.2f} {m['max_dd']:>6.1f}% {m['trades']:>7} {m['trades_yr']:>7.1f} {m['avg_days']:>6.0f} {m['win_rate']:>5.0f}% {m['avg_win']:>+7.1f}% {m['avg_loss']:>+7.1f}% {m['expectancy']:>+7.1f}%")

## 5. Fine-Grained Trail Stop Test (12H)

In [ ]:
print("="*140)
print("12H TIMEFRAME: FINE-GRAINED TRAIL STOP ANALYSIS")
print("="*140)
print(f"\n{'Trail':>6} {'Return':>10} {'CAGR':>8} {'Sharpe':>7} {'MaxDD':>7} {'Trades':>7} {'Tr/Yr':>7} {'Days':>6} {'Win%':>6} {'AvgWin':>8} {'AvgLoss':>8} {'Expect':>8}")
print("-"*140)

results_12h = {}

for trail in trail_stops:
    pf = run_strat002(df_12h, "12h", trail)
    if pf:
        m = get_metrics(pf, years)
        results_12h[trail] = {"m": m, "pf": pf}
        print(f"{trail*100:>5.0f}% {m['return']:>+9,.0f}% {m['cagr']:>+7.1f}% {m['sharpe']:>7.2f} {m['max_dd']:>6.1f}% {m['trades']:>7} {m['trades_yr']:>7.1f} {m['avg_days']:>6.0f} {m['win_rate']:>5.0f}% {m['avg_win']:>+7.1f}% {m['avg_loss']:>+7.1f}% {m['expectancy']:>+7.1f}%")

## 6. Income Optimization: Find Best Trade-off

In [ ]:
# Score each configuration for income generation
# Criteria: trades_yr * expectancy * sqrt(win_rate)

def income_score(m):
    """Score for income potential: frequency * edge * consistency"""
    if m["expectancy"] <= 0:
        return 0
    return m["trades_yr"] * m["expectancy"] * np.sqrt(m["win_rate"]/100)

print("="*120)
print("INCOME OPTIMIZATION: BEST CONFIGURATIONS")
print("="*120)

all_configs = []

for tf_name, results in [("4H", results_4h), ("8H", results_8h), ("12H", results_12h)]:
    for trail, data in results.items():
        m = data["m"]
        score = income_score(m)
        all_configs.append({
            "tf": tf_name,
            "trail": trail,
            "score": score,
            **m
        })

# Sort by income score
all_configs.sort(key=lambda x: x["score"], reverse=True)

print(f"\n{'Rank':>4} {'TF':>4} {'Trail':>6} {'Score':>8} {'Return':>10} {'Tr/Yr':>7} {'Days':>6} {'Win%':>6} {'Expect':>8} {'CAGR':>8}")
print("-"*120)

for i, c in enumerate(all_configs[:20]):
    print(f"{i+1:>4} {c['tf']:>4} {c['trail']*100:>5.0f}% {c['score']:>8.1f} {c['return']:>+9,.0f}% {c['trades_yr']:>7.1f} {c['avg_days']:>6.0f} {c['win_rate']:>5.0f}% {c['expectancy']:>+7.1f}% {c['cagr']:>+7.1f}%")

In [ ]:
# Highlight configurations with 10+ trades/year
print("\n" + "="*120)
print("HIGH FREQUENCY CONFIGS (10+ trades/year)")
print("="*120)

high_freq = [c for c in all_configs if c["trades_yr"] >= 10 and c["expectancy"] > 0]
high_freq.sort(key=lambda x: x["return"], reverse=True)

print(f"\n{'TF':>4} {'Trail':>6} {'Return':>10} {'CAGR':>8} {'Tr/Yr':>7} {'Days':>6} {'Win%':>6} {'Expect':>8} {'AvgWin':>8} {'AvgLoss':>8}")
print("-"*120)

for c in high_freq:
    print(f"{c['tf']:>4} {c['trail']*100:>5.0f}% {c['return']:>+9,.0f}% {c['cagr']:>+7.1f}% {c['trades_yr']:>7.1f} {c['avg_days']:>6.0f} {c['win_rate']:>5.0f}% {c['expectancy']:>+7.1f}% {c['avg_win']:>+7.1f}% {c['avg_loss']:>+7.1f}%")

## 7. Annual Income Projection

In [ ]:
# Project annual income for different capital levels
print("\n" + "="*100)
print("ANNUAL INCOME PROJECTION")
print("="*100)

capital_levels = [50000, 100000, 250000, 500000]

# Top 5 high-frequency configs
top_configs = high_freq[:5] if high_freq else all_configs[:5]

print(f"\n{'Config':<20} {'CAGR':>8}", end="")
for cap in capital_levels:
    print(f" {'$'+str(cap//1000)+'K':>12}", end="")
print()
print("-"*80)

for c in top_configs:
    config_name = f"{c['tf']} {c['trail']*100:.0f}%"
    print(f"{config_name:<20} {c['cagr']:>+7.1f}%", end="")
    for cap in capital_levels:
        annual = cap * c['cagr'] / 100
        print(f" ${annual:>10,.0f}", end="")
    print()

## 8. Trade Details for Best Config

In [ ]:
# Show trades for top income config
if high_freq:
    best = high_freq[0]
    best_tf = best["tf"]
    best_trail = best["trail"]
    
    if best_tf == "4H":
        pf = results_4h[best_trail]["pf"]
    elif best_tf == "8H":
        pf = results_8h[best_trail]["pf"]
    else:
        pf = results_12h[best_trail]["pf"]
    
    trades = pf.trades.records_readable
    
    print(f"\n{'='*100}")
    print(f"TRADE DETAILS: {best_tf} with {best_trail*100:.0f}% trail")
    print(f"{'='*100}")
    print(f"\nTotal trades: {len(trades)}")
    print(f"Trades/year: {best['trades_yr']:.1f}")
    print(f"Win rate: {best['win_rate']:.0f}%")
    print(f"\n{trades[['Entry Timestamp', 'Exit Timestamp', 'Return', 'PnL']].to_string()}")

## 9. Summary & Recommendation

In [ ]:
print("\n" + "="*100)
print("FINAL RECOMMENDATION FOR INCOME GENERATION")
print("="*100)

if high_freq:
    best = high_freq[0]
    print(f"""
BEST CONFIGURATION FOR ACTIVE INCOME:
    
    Timeframe:      {best['tf']}
    Trail Stop:     {best['trail']*100:.0f}%
    
    Trades/Year:    {best['trades_yr']:.1f}
    Avg Hold:       {best['avg_days']:.0f} days
    Win Rate:       {best['win_rate']:.0f}%
    
    CAGR:           {best['cagr']:+.1f}%
    Total Return:   {best['return']:+,.0f}%
    Max Drawdown:   {best['max_dd']:.1f}%
    
INCOME AT $100K CAPITAL:
    Annual:         ${100000 * best['cagr'] / 100:+,.0f}
    Per Trade:      ${100000 * best['expectancy'] / 100:+,.0f}
    
TRADE-OFF vs DAILY 30% TRAIL:
    - More trades ({best['trades_yr']:.0f}/yr vs ~2/yr)
    - Shorter holds ({best['avg_days']:.0f} days vs ~180 days)
    - Lower total return but more frequent opportunities
""")
else:
    print("\nNo high-frequency profitable configurations found.")
    print("Stick with daily STRAT-002 for swing trading.")